# Does weaker acid change what survives? Test one small peptide at two acid strengths

You'll pick one small peptide and decide whether it survives in 98% and in 81% sulfuric acid.

Pick a dipeptide nobody has posted yet in the lab's Discord channel, and post yours, so members
spread across all 18 (Ala-Ala and Gly-Gly are done in notebook 01).

The study behind notebook 01 tested 20 homodipeptides (two copies of one amino acid joined, such as
Ala-Ala) in 98% and in 81% sulfuric acid. After your hand-in is reviewed, your decisions for the two
acids are added to the shared results table in this repository, one line per acid, next to other
members' ([replication/README.md](https://github.com/Rakiura-Inc/morning-star-nmr/blob/main/replication/README.md)).
You are listed as "Rakiura member" unless you choose to be named.

| Part | What you do | Time |
|---|---|---|
| 0 | Pick a dipeptide and predict | 10 min |
| 1 | Stack both series and check them | 20 min |
| 2 | Compare first and last in each acid | 25 min |
| 3 | Record your decision | 10 min |
| 4 | Check your decision against the paper | 5 min |

There is no worked example for your molecule. You run the full analysis yourself, and Gly-Gly
from notebook 01 sits next to your answers as a filled example. Boxes headed **RECORD YOUR
ANSWER** are where you write something. You compare with the paper's tables at the end.

Data: Petkowski, M. D. Seager, Bains and S. Seager, Zenodo [10.5281/zenodo.11223995](https://doi.org/10.5281/zenodo.11223995), CC BY 4.0. Paper: *Scientific Reports* 14, 17083 (2024).

In [ ]:
# Setup. In Colab this first installs the training package and the Rakiura helper (about 30 s).
import importlib.util

if importlib.util.find_spec("nmrtrain") is None:
    %pip install -q "nmrtrain @ git+https://github.com/Rakiura-Inc/morning-star-nmr@main"
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"
import matplotlib.pyplot as plt
import nmrglue as ng  # the standard Python library for reading and processing NMR data
import numpy as np
from scipy.signal import find_peaks

import nmrtrain as nt  # this course's plumbing: downloads, checks, your answers, the hand-in

NOTEBOOK = "notebooks/02_one_dipeptide_both_acids.ipynb"
MODE = nt.portal.connect(NOTEBOOK)

> **CONNECT TO RAKIURA (optional, recommended)**
>
> **What it does:** ticks your checklist on the step as you go and attaches your figures to it. Nothing is sent until you connect.
>
> **How:** run the cell below, open the link it prints, check the code matches the one printed here, and click Allow. In Colab you do this once per session. To skip the click, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; get the key in Rakiura under Settings > Notebook key).
>
> **Skipping it is fine:** your work is kept on this computer, and you can attach your files on the step page instead.

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = nt.portal.connect(NOTEBOOK, sign_in=True)

## Part 0: Pick and predict

In [ ]:
DIPEPTIDE_SPECTRA = "11223995"  # Zenodo record: Petkowski et al., CC BY 4.0
for code in nt.data.samples(DIPEPTIDE_SPECTRA):
    print(code, nt.names.sample_name(code))

### What you need

From notebook 01 you know two decisions in 98% acid: Ala-Ala breaks down and Gly-Gly survives. Most
dipeptides break down in concentrated sulfuric acid (the summary card in the bootcamp's first step). 81% acid is 81% sulfuric acid
by weight and the rest heavy water (D2O), so it holds more water than 98% acid. Your prediction is a
guess with a reason; the spectra settle it.

> **RECORD YOUR ANSWER: pick a dipeptide and predict what the two acids do to it**
>
> **What gets saved:** `DIPEPTIDE`, the code you picked, and `MY_PREDICTION`, three short answers, saved with the time.
>
> **Where to look:** the list of codes above, and what you found in notebook 01.
>
> **How to fill this in:**
> - `DIPEPTIDE`: the two-letter code of the dipeptide you picked and posted in the lab's Discord channel, in quotes, like `"LL"`. Not `"AA"` or `"GG"`: notebook 01 covered those.
> - `"K"` (98% acid): `"survives"`, `"breaks down"` or `"unsure"`
> - `"H"` (81% acid): the same choices
> - `"harsher"`: which acid you expect to be harsher on it: `"98%"`, `"81%"`, `"same"` or `"unsure"`

<details><summary>Stuck? Open hint 1</summary>

Nothing in the structure alone settles this. Write a guess with a reason, and "unsure" where
you have no basis.

</details>

<details><summary>Stuck? Open hint 2</summary>

Two things pull in different directions: 98% acid is the stronger acid, but 81% acid holds more
water, and water is what splits a peptide bond on Earth. Pick the one you think wins for your
dipeptide.

</details>

In [ ]:
DIPEPTIDE = "LL"  # change this to the two-letter code of the dipeptide you picked

MY_PREDICTION = {
    "K": ...,  # 98% acid: "survives", "breaks down" or "unsure"
    "H": ...,  # 81% acid: "survives", "breaks down" or "unsure"
    "harsher": ...,  # which acid is harsher on it: "98%", "81%", "same" or "unsure"
}
nt.record.predict("dipeptide", MY_PREDICTION)

## Part 1: Load and stack

Acid code `K` is 98% and `H` is 81%. The next cell downloads both series (about 5 MB) and prints
where the acid peak sits in each spectrum.

In [ ]:
# How every spectrum in this notebook is read, with nmrglue.
def read_spectrum(zip_path):
    """The spectrum the instrument processed, with its ppm axis and labels (sample, acid, time)."""
    folder = nt.io.experiment_folder(zip_path)  # unzipped once, into nmr_data/
    # Bruker keeps the spectrum it processed in pdata/1; dic holds the instrument's settings.
    dic, data = ng.bruker.read_pdata(f"{folder}/pdata/1")
    # udic: the axis settings (number of points, spectral width, spectrometer frequency) from dic.
    udic = ng.bruker.guess_udic(dic, data)
    ppm = ng.fileiobase.uc_from_udic(udic).ppm_scale()  # the instrument's ppm scale, high to low
    return nt.io.labelled(zip_path, dic, ppm, data)

In [ ]:
files_98 = nt.data.fetch(DIPEPTIDE_SPECTRA, DIPEPTIDE, acid="98%")
files_81 = nt.data.fetch(DIPEPTIDE_SPECTRA, DIPEPTIDE, acid="81%")
spectra_98 = [read_spectrum(path) for path in files_98]  # first time point first
spectra_81 = [read_spectrum(path) for path in files_81]

In [ ]:
ACID_REGION_PPM = (10.0, 13.0)  # where the acid's own peak sits: 11.4 to 12.1 ppm in these data

In [ ]:
def tallest_ppm(spectrum, ppm_range):
    """Where the tallest point between two ppm values sits. In ACID_REGION_PPM: the acid's peak."""
    ppm, intensity = spectrum.region(*ppm_range)
    return float(ppm[np.argmax(intensity)])

In [ ]:
# The file names code 98% acid as K and 81% as H.
for code, spectra in (("K", spectra_98), ("H", spectra_81)):
    times = [spectrum.time_label for spectrum in spectra]
    acid_peaks = [round(tallest_ppm(spectrum, ACID_REGION_PPM), 2) for spectrum in spectra]
    print(code, times, "| acid peak at", acid_peaks)

In [ ]:
def scaled(spectrum, scale_region):
    """The intensity divided by its tallest point inside scale_region (low, high ppm), so that point
    is 1. With scale_region None, the tallest point anywhere (in a hydrogen spectrum in this acid,
    the acid's own peak)."""
    if scale_region is None:
        return spectrum.intensity / spectrum.intensity.max()
    low_ppm, high_ppm = scale_region
    _, intensity_in_region = spectrum.region(low_ppm, high_ppm)  # region gives (ppm, intensity)
    return spectrum.intensity / intensity_in_region.max()

In [ ]:
def plot_stack(spectra, ppm_range, *, scale_region, title, labels=None, ax=None, offset=1.2):
    """Traces stacked bottom to top in the order given, each scaled by scaled(), then raised by
    offset. Everything after the * is given by name: plot_stack(spectra, (0, 13), title=...)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 5))
    colours = plt.cm.viridis(np.linspace(0.0, 0.85, len(spectra)))  # dark to light, short of yellow
    for number, spectrum in enumerate(spectra):
        label = labels[number] if labels else spectrum.time_label
        trace = scaled(spectrum, scale_region) + offset * number
        ax.plot(spectrum.ppm, trace, lw=0.8, color=colours[number], label=label)
    ax.set_xlim(ppm_range[1], ppm_range[0])  # NMR convention: high ppm on the left
    ax.set_ylim(-0.2, offset * (len(spectra) - 1) + 1.4)
    ax.set_yticks([])  # the heights are scaled, so their numbers mean nothing
    ax.set_xlabel("chemical shift (ppm)")
    ax.set_title(title)
    handles, names = ax.get_legend_handles_labels()  # listed top trace first, like the stack
    ax.legend(handles[::-1], names[::-1], frameon=False, fontsize=8, loc="upper left")
    return ax.figure

**Stuck?** Gln-Gln (QQ) has no 98% four-month file: use 5 weeks, or pick another. His-His's
four-month files end in "4MonthR".

In 81% acid the acid peak sits at higher ppm (about 11.7 to 12.1 ppm, against 11.4 to 11.6 in 98%),
because its position depends on how much water the acid holds. Your dipeptide's own peaks also sit
at different ppm in the two acids. Compare each acid's series only with itself, never across acids.

Gly-Gly's peaks sit at 4.61 and 4.77 ppm in 98% acid and at 5.01 and 5.17 ppm in 81%. Val-Val's
methyl peaks sit about 0.25 ppm lower (further right) in 81%. Within one series the acid peak drifts
by up to about 0.1 ppm while the dipeptide's peaks stay put, so do not shift your files to line up
their acid peaks.

### What you need

`REGION` is the ppm range you plot and read. The hydrogens on carbon of most dipeptides sit
between 1 and 6.5 ppm. Hydrogens on an aromatic ring (a flat ring, in Phe-Phe, Tyr-Tyr, Trp-Trp
and His-His) sit between 6.5 and 9 ppm. Leave out the acid peak (11.4 to 12.1 ppm).

> **RECORD YOUR ANSWER: choose the ppm range to read**
>
> **What gets saved:** `REGION`, two numbers in ppm in round brackets, low then high.
>
> **Where to look:** the name of your dipeptide in the list above: does its side chain have an aromatic ring?
>
> **How to fill this in:**
> - No aromatic ring: `(1.0, 6.5)`. Aromatic ring: `(0.5, 9.5)`.
> - Left as `...`, the cell uses `(0.5, 9.5)`.

<details><summary>Stuck? Open hint 1</summary>

Look up your dipeptide's amino acid: is it Phe, Tyr, Trp or His?

</details>

<details><summary>Stuck? Open hint 2</summary>

Phe, Tyr, Trp and His have aromatic rings. Proline's ring is not aromatic, so Pro-Pro uses `(1.0, 6.5)`.

</details>

In [ ]:
REGION = ...  # two numbers in ppm, low then high, like (1.0, 6.5) or (0.5, 9.5)
REGION = nt.record.fill(REGION, (0.5, 9.5), "REGION")
nt.record.saved("REGION", REGION, "the ppm range you read")

name = nt.names.sample_name(DIPEPTIDE)
fig_stacks, (ax_98, ax_81) = plt.subplots(1, 2, figsize=(14, 5.5))
plot_stack(spectra_98, REGION, scale_region=REGION, title=f"{name} in 98% acid", ax=ax_98)
traps_98 = nt.checks.run_traps(spectra_98, norm=REGION, low_ppm=REGION[0], high_ppm=REGION[1])
plot_stack(spectra_81, REGION, scale_region=REGION, title=f"{name} in 81% acid", ax=ax_81)
traps_81 = nt.checks.run_traps(spectra_81, norm=REGION, low_ppm=REGION[0], high_ppm=REGION[1])
plt.show()
nt.portal.done("dipeptide-stack", figures=[fig_stacks])

### Look at every time point

**Predict:** which acid strength will be harsher on your dipeptide, and why? You recorded a guess in
Part 0; check it against the stacks now.

In each stack, note the first time point where something changes. Look at the middle time points
too, not only day 1 and four months: the paper often reports a dipeptide as "stable for at least 5
weeks", and only the 5 week file can show that.

Some dipeptides may change before the first measurement. If day 1 already shows more peaks than your
dipeptide should have, compare it with the lab's reference spectrum of the same dipeptide in heavy
water (water with deuterium in place of hydrogen): "Suplementary dataset S2.zip" from the same
Zenodo record as S1, file SZSSea-XX-A-1Day-1H, with your code in place of XX. Count the peaks from
hydrogens on carbon in each. Peaks sit at different ppm in water and in acid, so compare their
number and pattern, not their positions.

## Part 2: First vs last, in each acid

In some samples the whole spectrum moves a little between the first and last time points,
especially in 81% acid. A peak that moved by the same amount as all the others is the same peak.
The next cell lists the peaks kept, lost and new in each acid.

In [ ]:
def group_lines(lines, group_ppm):
    """Lines (ppm, height), low ppm first, closer than group_ppm to the line before belong to the
    same signal: a methyl's two lines are 0.016 ppm apart."""
    groups = []
    previous_ppm = None
    for ppm, height in lines:
        if previous_ppm is not None and ppm - previous_ppm < group_ppm:
            groups[-1].append((ppm, height))
        else:
            groups.append([(ppm, height)])
        previous_ppm = ppm
    return groups

In [ ]:
def signal_from(lines):
    """One signal from its lines: their height-weighted centre, how many, and the tallest."""
    total_height = sum(height for _, height in lines)
    centre_ppm = sum(ppm * height for ppm, height in lines) / total_height
    tallest = max(height for _, height in lines)
    return {
        "ppm": round(float(centre_ppm), 3),
        "lines": len(lines),
        "height": round(float(tallest), 3),
    }

In [ ]:
def pick_peaks(spectrum, ppm_range, *, min_height, group_ppm=0.03, scale_region=None):
    """The signals between two ppm values, high ppm first, each {"ppm", "lines", "height"}.
    min_height and the heights are shares of the tallest point in ppm_range, or in scale_region
    when you give one (to look for small peaks in a quiet window)."""
    ppm, intensity = spectrum.region(*ppm_range)  # the (low, high) pair, unpacked
    if scale_region is None:
        heights = intensity / intensity.max()
    else:
        _, intensity_in_region = spectrum.region(*scale_region)
        heights = intensity / intensity_in_region.max()
    # A line must reach min_height and stand 0.02 above the dips beside it (ripples do not).
    found, _ = find_peaks(heights, height=min_height, prominence=0.02)
    lines = sorted(zip(ppm[found], heights[found]))  # (ppm, height) pairs, low ppm first
    signals = [signal_from(group) for group in group_lines(lines, group_ppm)]
    return signals[::-1]  # high ppm first, the way a spectrum is read

In [ ]:
MIN_HEIGHT = 0.03  # the smallest peak picked, as a share of the tallest in REGION

first_98, last_98 = spectra_98[0], spectra_98[-1]
first_81, last_81 = spectra_81[0], spectra_81[-1]
peaks_first_98 = pick_peaks(first_98, REGION, min_height=MIN_HEIGHT)
peaks_last_98 = pick_peaks(last_98, REGION, min_height=MIN_HEIGHT)
peaks_first_81 = pick_peaks(first_81, REGION, min_height=MIN_HEIGHT)
peaks_last_81 = pick_peaks(last_81, REGION, min_height=MIN_HEIGHT)

In [ ]:
SAME_PEAK_PPM = 0.05  # two peaks this close, at two times, are the same peak


def compare_peaks(first_ppm, last_ppm, tolerance_ppm=SAME_PEAK_PPM):
    """Which peaks are at both times (kept), at the first only (lost) and at the last only
    (new). Two peaks within tolerance_ppm of each other count as the same peak."""
    changes = {"kept": [], "lost": [], "new": []}
    for ppm in first_ppm:
        if any(abs(ppm - later) <= tolerance_ppm for later in last_ppm):
            changes["kept"].append(ppm)
        else:
            changes["lost"].append(ppm)
    for ppm in last_ppm:
        if not any(abs(ppm - earlier) <= tolerance_ppm for earlier in first_ppm):
            changes["new"].append(ppm)
    return changes

In [ ]:
ppm_first_98 = [peak["ppm"] for peak in peaks_first_98]
ppm_last_98 = [peak["ppm"] for peak in peaks_last_98]
ppm_first_81 = [peak["ppm"] for peak in peaks_first_81]
ppm_last_81 = [peak["ppm"] for peak in peaks_last_81]
changes_98 = compare_peaks(ppm_first_98, ppm_last_98)
changes_81 = compare_peaks(ppm_first_81, ppm_last_81)

In [ ]:
def listed(positions_ppm):
    """Positions as '1.99, 2.08', or 'none'."""
    if not positions_ppm:
        return "none"
    return ", ".join(f"{ppm:.2f}" for ppm in positions_ppm)

In [ ]:
for strength, changes in (("98%", changes_98), ("81%", changes_81)):
    kept, lost, new = listed(changes["kept"]), listed(changes["lost"]), listed(changes["new"])
    print(f"{strength}: at both times {kept} | first only {lost} | last only {new}")

In [ ]:
COMPARE_OFFSET = 1.15  # how far up the later trace is drawn, so the two do not overlap


def mark_peaks(ax, peaks, baseline):
    """Write each peak's position above it. baseline: the height its trace is raised by."""
    for peak in peaks:
        top = (peak["ppm"], peak["height"] + baseline)
        label = f"{peak['ppm']:.2f}"
        ax.annotate(
            label,
            top,
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            fontsize=7.5,
            rotation=90,
        )

In [ ]:
def plot_first_vs_last(ax, first, last, *, first_peaks, last_peaks, title):
    """The first spectrum (bottom) and the last (top), scaled within REGION, peaks labelled."""
    plot_stack(
        [first, last], REGION, scale_region=REGION, title=title, ax=ax, offset=COMPARE_OFFSET
    )
    mark_peaks(ax, first_peaks, baseline=0.0)
    mark_peaks(ax, last_peaks, baseline=COMPARE_OFFSET)
    ax.set_ylim(-0.1, COMPARE_OFFSET + 1.6)  # room for the labels over the top trace

In [ ]:
fig_cmp, (ax_98, ax_81) = plt.subplots(1, 2, figsize=(14, 5))
title_98 = f"98%: {first_98.time_label} vs {last_98.time_label}"
title_81 = f"81%: {first_81.time_label} vs {last_81.time_label}"
plot_first_vs_last(
    ax_98, first_98, last_98, first_peaks=peaks_first_98, last_peaks=peaks_last_98, title=title_98
)
plot_first_vs_last(
    ax_81, first_81, last_81, first_peaks=peaks_first_81, last_peaks=peaks_last_81, title=title_81
)
plt.show()

### How much is left

Measure one of your dipeptide's peaks against the acid peak in the same file, then divide that ratio
by its value at the first time point:

- Under one third: little or none is left. This is clear loss.
- One third or more: no clear loss by this measure. Decide from the spectra: whether new peaks
  appear, and whether your dipeptide's peaks keep their sizes relative to each other.

Raw areas cannot be compared between files, because each file's intensity is scaled differently.
Dividing by all the peaks from hydrogens on carbon makes too much look left, because that total
shrinks as the dipeptide breaks down: for Ala-Ala at 8 days it gives 89% left, where the acid peak
gives 57%. The acid peak is not a fixed yardstick either. It collects the hydrogens the sample hands
over, so its size changes over time. In dipeptides the paper calls stable, a peak measured against
it still drifts between about half and one and a half times its day 1 value (His-His in 98% acid
ends at about 0.5 to 0.65 at four months). So only a fall below one third is clear loss.

The same measurement on Ala-Ala in 98% acid (notebook 01): Ala-Ala looks unchanged at 1 day. The
change first shows at 3 days, and the loss is gradual: 91% is left at 3 days, 57% at 8 days, 7% at 5
weeks and none at 4 months. The 3 and 8 day values alone would be inside the acid peak's drift, but
the alpha at 5.17 ppm, measured against the alpha at 4.92 ppm in the same file, falls in step (85% at
3 days, 69% at 8 days).

The next cell follows the tallest peak from hydrogens on carbon at the first time point, and also
measures it against your dipeptide's next tallest peak, which does not use the acid peak. Follow
peaks that the intact dipeptide has at the first time point and that no later peak overlaps. If your
stack shows a new peak growing on top of one, set `SIGNAL_PPM` or `OTHER_PPM` for that acid to another
of your dipeptide's peaks and run the cell again.

In [ ]:
# Optional, per acid: the ppm of the peak to follow, like 1.41. None takes the tallest.
SIGNAL_PPM = {"98%": None, "81%": None}
# Optional, per acid: a second peak of your dipeptide. None takes the next tallest.
OTHER_PPM = {"98%": None, "81%": None}

# Follow hydrogens on carbon only. Above 6.3 ppm the peaks are hydrogens on nitrogen (NH,
# NH3+), unless the dipeptide has an aromatic ring: Phe-Phe, Tyr-Tyr, Trp-Trp, His-His.
NITROGEN_HYDROGENS_FROM_PPM = 6.3
AROMATIC_DIPEPTIDES = ("Phe-Phe", "Tyr-Tyr", "Trp-Trp", "His-His")
if nt.names.sample_name(DIPEPTIDE) in AROMATIC_DIPEPTIDES:
    follow_ppm = REGION
else:
    follow_ppm = (REGION[0], min(REGION[1], NITROGEN_HYDROGENS_FROM_PPM))

In [ ]:
def peak_area(spectrum, ppm_range):
    """The sum of the intensity between two ppm values: a rough area, enough to compare peaks
    within one file."""
    _, intensity = spectrum.region(*ppm_range)
    return float(intensity.sum())

In [ ]:
def area_ratios(spectra, peak_ppm, reference_ppm):
    """At each time point, the area of one peak (peak_ppm: its low and high edges) divided by
    the area of a reference peak in the same file."""
    ratios = []
    for spectrum in spectra:
        ratio = peak_area(spectrum, peak_ppm) / peak_area(spectrum, reference_ppm)
        ratios.append(max(ratio, 0.0))  # where a peak has gone, noise can sum below zero
    return ratios


def share_of_first(ratios):
    """Each ratio as a fraction of the first one: 1.0 at the first time point."""
    if ratios[0] == 0:
        return [0.0 for _ in ratios]  # a peak you chose is not there at the first time point
    return [ratio / ratios[0] for ratio in ratios]

In [ ]:
CLEAR_LOSS_BELOW = 1 / 3  # under a third of day 1 is clear loss (see How much is left)


def how_much_is_left(fraction):
    """The two levels the steps use, from the fraction left at the last time point."""
    if fraction < CLEAR_LOSS_BELOW:
        return "little or none"
    return "no clear loss"

In [ ]:
def next_tallest_ppm(spectrum, ppm_range, away_from_ppm, gap_ppm=0.1):
    """The tallest picked peak at least gap_ppm from away_from_ppm, or None if there is none."""
    tallest = None
    for peak in pick_peaks(spectrum, ppm_range, min_height=MIN_HEIGHT):
        far_enough = abs(peak["ppm"] - away_from_ppm) >= gap_ppm
        if far_enough and (tallest is None or peak["height"] > tallest["height"]):
            tallest = peak
    if tallest is None:
        return None
    return tallest["ppm"]

In [ ]:
FOLLOW_HALF_WIDTH_PPM = 0.05  # the window taken either side of a followed peak


def follow_peak(spectra, peak_ppm, other_ppm):
    """One peak (FOLLOW_HALF_WIDTH_PPM either side of peak_ppm) over the series: its area
    against the acid peak, that as a share of the first time point, and the same against another
    of its own peaks (an empty list when there is no other peak)."""
    window = (peak_ppm - FOLLOW_HALF_WIDTH_PPM, peak_ppm + FOLLOW_HALF_WIDTH_PPM)
    against_acid = area_ratios(spectra, window, ACID_REGION_PPM)
    followed = {"peak_ppm": peak_ppm, "other_ppm": other_ppm, "against_other": []}
    followed["ratios"] = against_acid
    followed["left"] = share_of_first(against_acid)
    if other_ppm:
        other_window = (other_ppm - FOLLOW_HALF_WIDTH_PPM, other_ppm + FOLLOW_HALF_WIDTH_PPM)
        followed["against_other"] = share_of_first(area_ratios(spectra, window, other_window))
    return followed

In [ ]:
followed = {}  # by acid code: K is 98%, H is 81%
for code, strength, spectra in (("K", "98%", spectra_98), ("H", "81%", spectra_81)):
    peak_ppm = SIGNAL_PPM[strength]
    if peak_ppm is None:  # not chosen: follow the tallest
        peak_ppm = tallest_ppm(spectra[0], follow_ppm)
    other_ppm = OTHER_PPM[strength]
    if other_ppm is None:  # not chosen: the next tallest, at least 0.1 ppm away
        other_ppm = next_tallest_ppm(spectra[0], follow_ppm, away_from_ppm=peak_ppm)
    followed[code] = follow_peak(spectra, peak_ppm, other_ppm)

In [ ]:
for code, strength, spectra in (("K", "98%", spectra_98), ("H", "81%", spectra_81)):
    peak = followed[code]
    first, last = spectra[0].time_label, spectra[-1].time_label
    ratios, left = peak["ratios"], peak["left"]
    shares = ", ".join(
        f"{spectrum.time_label} {share:.0%}" for spectrum, share in zip(spectra, left)
    )
    print(
        f"{strength}: peak near {peak['peak_ppm']:.2f} ppm divided by the acid peak, "
        f"as a percentage of its {first} value: {shares}"
    )
    band = how_much_is_left(left[-1])
    division = f"{ratios[-1]:.2f} at {last} / {ratios[0]:.2f} at {first}"
    print(f'     {division} = {left[-1]:.0%}, so "{band}"')
    if peak["against_other"]:
        against_other = zip(spectra, peak["against_other"])
        other_shares = ", ".join(
            f"{spectrum.time_label} {share:.0%}" for spectrum, share in against_other
        )
        other_ppm = peak["other_ppm"]
        instead = f"against your peak near {other_ppm:.2f} ppm instead (no acid peak)"
        print(f"     {instead}: {other_shares}")

In [ ]:
LEFT_SEEN = {}  # what you measured, by acid code, for your hand-in
for code, peak in followed.items():
    LEFT_SEEN[code] = {
        "peak_ppm": round(peak["peak_ppm"], 2),
        "ratio_first": round(peak["ratios"][0], 3),
        "ratio_last": round(peak["ratios"][-1], 3),
        "left": round(peak["left"][-1], 2),
        "band": how_much_is_left(peak["left"][-1]),
        "other_ppm": None,
        "against_other_last": None,
    }
    if peak["against_other"]:
        LEFT_SEEN[code]["other_ppm"] = round(peak["other_ppm"], 2)
        LEFT_SEEN[code]["against_other_last"] = round(peak["against_other"][-1], 2)

### A filled example: Gly-Gly in 98% acid

Gly-Gly from notebook 01, filled in with the reasons. It shows the format; your dipeptide's values
will differ.

Gly-Gly survives at least 4 months in 98% acid (Table 2 of the paper). One of its two CH2 peaks, near
4.77 ppm, still weakens: by 4 months it is about a third of its day 1 size against the acid peak,
while the 4.61 ppm peak keeps most of its size and no sizable new peaks appear. The paper does not
explain this, so do not count it as breakdown.

In [ ]:
GLY_GLY_EXAMPLE = {
    "call": "survives",  # both CH2 signals are still there, in the same place, at 4 months
    "lost_ppm": [],  # nothing from day 1 is gone
    # a tiny peak that grows from about 8 days on, far smaller than the two main signals
    "new_ppm": [4.33],
    "kept_ppm": [4.77, 4.61],  # the two CH2 signals, one per glycine
    # both CH2 peaks keep a third or more of their day 1 value against the acid peak
    "left": "no clear loss",
    "product_guess": "none",
    "confidence": "high",
    "note": "notebook 01, for comparison",
}

### What you need

- **Lost:** a peak at the first time point that is gone, or much smaller, at the last.
- **New:** a peak at the last time point that was not there at the first.
- **Kept:** a peak at both times. A peak that moved by the same amount as all the others is the same
  peak.
- **How much is left:** from the percentages printed above: "little or none" or "no clear loss".
- Peaks of hydrogens on nitrogen (NH) fade as those hydrogens swap with the acid's deuterium. That
  is not breakdown. The peaks of hydrogens next to them sharpen after day 1, because they are
  no longer split by them. That is not breakdown either.

> **RECORD YOUR ANSWER: what changed in each acid?**
>
> **What gets saved:** `K_LOST_PPM`, `K_NEW_PPM`, `K_KEPT_PPM` and `K_LEFT` for 98% acid, and the same four starting `H_` for 81% acid.
>
> **Where to look:** the two first-vs-last plots above (98% on the left, 81% on the right), the lists printed under them, and the percentages under How much is left.
>
> **How to fill this in:**
> - `..._LOST_PPM`: peaks there at the first time point and gone or much smaller at the last, as numbers like `[1.23, 4.56]`
> - `..._NEW_PPM`: peaks at the last time point that were not there at the first
> - `..._KEPT_PPM`: peaks there at both times
> - `..._LEFT`: `"little or none"`, `"no clear loss"` or `"unsure"`
> - Leave a list as `[]` if there were none. Gly-Gly's record above shows the format.

<details><summary>Stuck? Open hint 1</summary>

Read one acid at a time, one peak at a time: for each first-time peak, is there still a peak in the same place at the last time point?

</details>

<details><summary>Stuck? Open hint 2</summary>

In 81% acid the whole spectrum can move a little between time points. If every peak moved by
about the same amount, count them as kept.

</details>

In [ ]:
K_LOST_PPM = []  # 98% acid: peaks at the first time point, gone or much smaller at the last
K_NEW_PPM = []  # 98% acid: peaks at the last time point that were not there at the first
K_KEPT_PPM = []  # 98% acid: peaks there at both times
K_LEFT = ""  # 98% acid: "little or none", "no clear loss" or "unsure"
H_LOST_PPM = []  # 81% acid: the same four
H_NEW_PPM = []
H_KEPT_PPM = []
H_LEFT = ""

in_98, in_81 = f"{name} in 98% acid", f"{name} in 81% acid"
nt.record.write_down("K_LOST_PPM", K_LOST_PPM, in_98, "lost_ppm", suggestion=changes_98["lost"])
nt.record.write_down("K_NEW_PPM", K_NEW_PPM, in_98, "new_ppm", suggestion=changes_98["new"])
nt.record.write_down("K_KEPT_PPM", K_KEPT_PPM, in_98, "kept_ppm", suggestion=changes_98["kept"])
nt.record.write_down("K_LEFT", K_LEFT, in_98, "left")
nt.record.write_down("H_LOST_PPM", H_LOST_PPM, in_81, "lost_ppm", suggestion=changes_81["lost"])
nt.record.write_down("H_NEW_PPM", H_NEW_PPM, in_81, "new_ppm", suggestion=changes_81["new"])
nt.record.write_down("H_KEPT_PPM", H_KEPT_PPM, in_81, "kept_ppm", suggestion=changes_81["kept"])
nt.record.write_down("H_LEFT", H_LEFT, in_81, "left")

## Part 3: Your decision

### What you need

For each acid, pick one:
- **Survives:** every peak from hydrogens on carbon stays in place and keeps its size relative to the
  molecule's other peaks, and no new peaks appear. By the measure above, the last value is one third
  or more of the first value.
- **Side chain changed:** the backbone peaks stay, but the side chain reacts. New peaks with the same
  pattern appear close to the old ones.
- **Breaks down:** peaks from hydrogens on carbon shrink or vanish while peaks from the products
  appear and grow.
- **Unsure:** the data do not settle it. Say why in your note.

If it broke down, name the main product if you can. Splitting the peptide bond with water
(hydrolysis) would give the single amino acid: compare the new peaks with its own spectrum in the
same acid ([Zenodo 8381013](https://zenodo.org/records/8381013)). If they match, name it. If
they do not, write "not the single amino acid" and give the new peaks' positions. In 98% acid the product can differ:
Ala-Ala gave alaninamide (notebook 01).

Two hydrogen spectra from separate samples match when the peaks have the same pattern (number of
lines and their spacing) and sit within 0.05 ppm of each other. From 0.05 to 0.1 ppm apart, hydrogen
alone does not settle it: compare carbon spectra, where within 0.1 ppm is a match. More than 0.1
ppm apart is not a match. The same molecule sits 0.01 to 0.07 ppm apart between samples in this data.

> **RECORD YOUR ANSWER: your decision in each acid**
>
> **What gets saved:** `K_CALL`, `H_CALL`, `K_PRODUCT`, `H_PRODUCT`, `K_CONFIDENCE`, `H_CONFIDENCE`, `HARSHER` and `NOTE`.
>
> **Where to look:** the lost, new and kept peaks you saved above, and how much is left.
>
> **How to fill this in:**
> - `..._CALL`: your decision in that acid, one of the four above: `"survives"`, `"side chain changed"`, `"breaks down"` or `"unsure"`
> - `..._PRODUCT`: the main product in words, `"none"`, or `""` if you have no idea yet
> - `..._CONFIDENCE`: how sure you are: `"low"`, `"medium"` or `"high"`
> - `HARSHER`: the acid that was harsher on it: `"98%"`, `"81%"`, `"same"` or `"unsure"`
> - `NOTE`: the first time point where you saw a change in each acid, and anything else your reviewer should know

<details><summary>Stuck? Open hint 1</summary>

Which acid lost more of its first-time peaks, and in which is less left against the acid peak?

</details>

<details><summary>Stuck? Open hint 2</summary>

If one acid shows "little or none" and the other "no clear loss", the first was harsher. If both look
the same, write "same". For the first change, look at the middle time points in your stacks.

</details>

In [ ]:
K_CALL = ""  # 98%, your decision: "survives", "side chain changed", "breaks down" or "unsure"
H_CALL = ""  # 81%: the same choices
K_PRODUCT = ""  # 98%: the main product in words, "none", or "" if no idea yet
H_PRODUCT = ""  # 81%: the same
K_CONFIDENCE = ""  # 98%: "low", "medium" or "high"
H_CONFIDENCE = ""  # 81%: "low", "medium" or "high"
HARSHER = ""  # "98%", "81%", "same" or "unsure"
NOTE = ""  # the first time point with a change in each acid, and anything else for your reviewer

nt.record.write_down("K_CALL", K_CALL, in_98, "call")
nt.record.write_down("K_PRODUCT", K_PRODUCT, in_98, "product_guess")
nt.record.write_down("K_CONFIDENCE", K_CONFIDENCE, in_98, "confidence")
nt.record.write_down("H_CALL", H_CALL, in_81, "call")
nt.record.write_down("H_PRODUCT", H_PRODUCT, in_81, "product_guess")
nt.record.write_down("H_CONFIDENCE", H_CONFIDENCE, in_81, "confidence")
nt.record.write_down("HARSHER", HARSHER, name, "harsher")
nt.record.write_down("NOTE", NOTE, name, "note")

### Your verdict

Your verdict is everything you recorded, in one table with a line per acid: your decision, the peaks
behind it, how much is left, the product and how sure you are. To change it, edit a line above and
run this cell again. The Gly-Gly line is the filled example from notebook 01.

In [ ]:
# Your answers for each acid, gathered from the cells above.
verdict_98 = {
    "call": K_CALL,
    "lost_ppm": K_LOST_PPM,
    "new_ppm": K_NEW_PPM,
    "kept_ppm": K_KEPT_PPM,
    "left": K_LEFT,
    "product_guess": K_PRODUCT,
    "confidence": K_CONFIDENCE,
}
verdict_81 = {
    "call": H_CALL,
    "lost_ppm": H_LOST_PPM,
    "new_ppm": H_NEW_PPM,
    "kept_ppm": H_KEPT_PPM,
    "left": H_LEFT,
    "product_guess": H_PRODUCT,
    "confidence": H_CONFIDENCE,
}

In [ ]:
MY_VERDICT = nt.record.build_verdict(
    {"K": verdict_98, "H": verdict_81},
    names={"K": f"{name}, 98%", "H": f"{name}, 81%"},
    examples={"Gly-Gly, 98% (example)": GLY_GLY_EXAMPLE},
    harsher=HARSHER,
    note=NOTE,
)

## Part 4: Check your decision against the paper

The paper gives its decisions in two tables: Table 1 for 81% acid and Table 2 for 98% acid (Petkowski
et al. 2024, *Scientific Reports* 14, 17083; [open full text](https://pmc.ncbi.nlm.nih.gov/articles/PMC11269616/)).
Each gives how long a dipeptide stays stable and what happens to it. Find yours in both, and compare
with your decision and the first time point where you saw a change.

The next cell holds the paper's decisions for Ala-Ala and Gly-Gly in 98% acid only, so for your
dipeptide it points you to the tables. Write what you found in `NOTE` in Part 3, then run the cells
from there down again. If your decision differs from the paper's, look again at the peaks you
listed; your hand-in goes to review as you wrote it.

In [ ]:
feedback = {}  # by acid code, as in MY_VERDICT
matches = {}  # does your decision match the paper's: "yes", "no" or "unsure"
if nt.record.revealed("dipeptide"):
    key = nt.answers.load("dipeptides")
    for code, strength in (("K", "98%"), ("H", "81%")):
        published = nt.answers.case(key, f"{DIPEPTIDE}-{code}")
        feedback[code] = nt.checks.check_verdict(MY_VERDICT[code], published)
        nt.checks.show(feedback[code], f"Your verdict, {strength} acid")
        matches[code] = nt.checks.matches_published(MY_VERDICT[code], published)

In [ ]:
# Your two lines for the shared results table, one per acid (saved with your hand-in).
traps_count = nt.checks.traps_flagged(traps_98, traps_81)
times_98 = [first_98.time, last_98.time]
times_81 = [first_81.time, last_81.time]
matches_98 = matches.get("K", "unsure")
matches_81 = matches.get("H", "unsure")
row_98 = nt.record.replication_row(
    DIPEPTIDE, "K", MY_VERDICT["K"], times_98, matches=matches_98, traps=traps_count
)
row_81 = nt.record.replication_row(
    DIPEPTIDE, "H", MY_VERDICT["H"], times_81, matches=matches_81, traps=traps_count
)

## Hand-in

> **SAVE YOUR HAND-IN: your dipeptide hand-in**
>
> **What gets saved:** in `handin/`: the stacks for both acids, the first vs last comparison, `verdict.json` (your verdict table, how much is left, peak lists and feedback) and `replication_rows.csv` (your two lines for the shared results table, one per acid).
>
> **Where to look:** nothing new to fill in: the cell uses what you recorded above.
>
> **How to fill this in:**
> - Run the cell. These files are your hand-in for the NMR project step "Does weaker acid change what survives?": the stacks are your two stacked spectra, and `verdict.json` holds your note.
> - Nothing is sent for review until you run `nt.portal.submit()` in the last line of the cell.
> - In Colab: open the folder icon on the left, then `handin/dipeptide-verdict`, and download each file before you close the tab.

<details><summary>Stuck? Open hint 1</summary>

If a figure is missing from the list it prints, run the cell that draws it, then this cell again.

</details>

<details><summary>Stuck? Open hint 2</summary>

If it says a name is not defined, run the notebook from the top (Runtime > Run all in Colab).

</details>

In [ ]:
record = nt.record.handin(
    "dipeptide-verdict",
    figures={"stacks": fig_stacks, "first_vs_last": fig_cmp},
    verdict={
        "dipeptide": DIPEPTIDE,
        "verdict": MY_VERDICT,
        "left_vs_acid_peak": LEFT_SEEN,
        "my_peaks_ppm": {
            "K": {first_98.time: ppm_first_98, last_98.time: ppm_last_98},
            "H": {first_81.time: ppm_first_81, last_81.time: ppm_last_81},
        },
        "feedback": feedback,
    },
    numbers={
        "sample": DIPEPTIDE,
        "call_98": MY_VERDICT["K"]["call"],
        "call_81": MY_VERDICT["H"]["call"],
        "harsher": MY_VERDICT["harsher"],
    },
    question="dipeptide",
    traps=traps_count,
)
nt.record.save_rows([row_98, row_81], "dipeptide-verdict")
# nt.portal.submit()